# CODI Training on GPU - Simplified Version

**No CODI repo cloning needed!** Everything is bundled.

**GPU**: Free T4 → Runtime → Change runtime type → T4 GPU

**Time**: ~2 hours for 3 epochs on 6,000 examples

In [1]:
# Verify GPU is active
import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

✅ GPU: Tesla T4
✅ Memory: 15.6 GB


In [2]:
# Clone your repo (includes codi_bundle with everything needed)
!git clone https://github.com/nabilanewaz/TokenSkip.git
%cd TokenSkip
!ls -lh codi_bundle/

Cloning into 'TokenSkip'...
remote: Enumerating objects: 322, done.
remote: Counting objects: 100% (204/204), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 322 (delta 59), reused 172 (delta 38), pack-reused 118 (from 1)
Receiving objects: 100% (322/322), 12.44 MiB | 8.62 MiB/s, done.
Resolving deltas: 100% (104/104), done.
/content/TokenSkip
total 32K
-rw-r--r-- 1 root root  765 Feb 25 15:23 README.md
drwxr-xr-x 2 root root 4.0K Feb 25 15:23 src
-rw-r--r-- 1 root root  21K Feb 25 15:23 train.py


In [3]:
# Install dependencies
!pip install peft==0.15.2 datasets==3.6.0 transformers==4.52.4 accelerate==1.7.0 safetensors -q
print("✅ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 136.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 86.3 MB/s eta 0:00:00
✅ Dependencies installed


In [4]:
# Verify training data
import json
train_file = 'datasets/gsm8k_split/llm_train.jsonl'
with open(train_file) as f:
    train_count = sum(1 for _ in f)
print(f"✅ Training data: {train_count} examples")

# Show first example
with open(train_file) as f:
    example = json.loads(f.readline())
print(f"\nExample: {example['question'][:100]}...")
print(f"Has CoT: {'cot' in example}")

✅ Training data: 6000 examples

Example: In a conference room, 40 chairs with a capacity of 2 people each were arranged in rows in preparatio...
Has CoT: True


In [5]:
# Download CODI checkpoint (GPT-2 + latent projection)
from huggingface_hub import snapshot_download
import os

ckpt_dir = snapshot_download(
    repo_id="zen-E/CODI-gpt2",
    ignore_patterns=["*.msgpack", "*.h5"]
)
print(f"✅ Checkpoint: {ckpt_dir}")

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md:   0%|          | 0.00/218 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/406M [00:00<?, ?B/s]

✅ Checkpoint: /root/.cache/huggingface/hub/models--zen-E--CODI-gpt2/snapshots/fd641b3d3edc59e4f534b55588e906588c9e36bb


In [6]:
# Prepare training data in CODI format
%cd codi_bundle

# Create datasets folder
!mkdir -p datasets/gsm8k
!cp ../datasets/gsm8k_split/llm_train.jsonl datasets/gsm8k/train.jsonl
!cp ../datasets/gsm8k_split/validation.jsonl datasets/gsm8k/val.jsonl
print("✅ Data prepared")

/content/TokenSkip/codi_bundle
✅ Data prepared


In [ ]:
# Start training (adjust batch_size based on GPU memory)
# T4: batch_size=4, A100: batch_size=8+
!python train.py \
  --model_name_or_path gpt2 \
  --seed 42 \
  --model_max_length 512 \
  --lora_r 128 \
  --lora_alpha 32 \
  --lora_init \
  --num_latent 6 \
  --use_prj True \
  --prj_dim 768 \
  --inf_latent_iterations 6 \
  --remove_eos True \
  --use_lora True \
  --per_device_train_batch_size 4 \
  --data_name custom_local:datasets/gsm8k/train.jsonl \
  --output_dir ../outputs/codi_trained \
  --num_train_epochs 3 \
  --learning_rate 0.0002

Streaming output truncated to the last 5000 lines.
loss=1.794921875, ce_loss=0.982421875, distill_loss=0.8125, ce_loss_total=0.982421875, distill_loss_total=0.8125, ref_ce_loss=1.369140625
 37% 1686/4500 [15:17<23:59,  1.95it/s]latent5: distill_loss=0.74267578125
loss=1.58984375, ce_loss=0.8466796875, distill_loss=0.74267578125, ce_loss_total=0.8466796875, distill_loss_total=0.74267578125, ref_ce_loss=1.3798828125
 37% 1687/4500 [15:17<23:42,  1.98it/s]latent5: distill_loss=0.681640625
loss=1.50390625, ce_loss=0.822265625, distill_loss=0.681640625, ce_loss_total=0.822265625, distill_loss_total=0.681640625, ref_ce_loss=1.392578125
 38% 1688/4500 [15:18<23:56,  1.96it/s]latent5: distill_loss=0.6484375
loss=1.5888671875, ce_loss=0.9404296875, distill_loss=0.6484375, ce_loss_total=0.9404296875, distill_loss_total=0.6484375, ref_ce_loss=1.6875
 38% 1689/4500 [15:18<23:48,  1.97it/s]latent5: distill_loss=0.740234375
loss=1.556640625, ce_loss=0.81689453125, distill_loss=0.740234375, ce_loss_t

In [ ]:
# Monitor progress (run this cell repeatedly)
!tail -n 30 ../outputs/codi_trained/*/logs.txt 2>/dev/null || echo "Check for log files in outputs/"

In [ ]:
# After training: check what was saved
%cd ..
!ls -lh outputs/codi_trained/

In [ ]:
# Download checkpoint
!zip -r codi_checkpoint.zip outputs/codi_trained/
from google.colab import files
files.download('codi_checkpoint.zip')
print("✅ Checkpoint download started!")

## Next: Phase 2 - Truth Vector Extraction

After downloading the checkpoint:

```powershell
# Extract locally
Expand-Archive codi_checkpoint.zip

# Phase 2: Extract truth vector from 500 steer examples
python extract_truth_vector.py --steer-data datasets/gsm8k_split/steer_train.jsonl
```